# End2Race PPO GAE diagnostic

## tl;dr

No GAE correctness defect is visible. Collision-role transitions carry 5.83x the ordinary-role raw
advantage second-moment proxy in U20-U30, but the proxy does not positively track actor KL or gradient
spikes. Keep lambda at 0.995 until transition-level counterfactual telemetry can test 0.99 on the same rollout.


## Context & Methods

The focal run is `ppo_privilege_gru_0722_long_clip020`: gamma 0.999, lambda 0.995, 30 updates,
12 workers, and a 100 Hz simulator. The notebook loads bounded CSVs produced from the persisted run
config, `metrics.jsonl`, and `episodes.jsonl` by `analyze_gae.py`.

### Key Assumptions

- Before critic training, replayed values closely match values stored during collection, so pre-update
  value MSE is a proxy for the raw advantage second moment, not a direct saved distribution.
- Correlations across 29 updates are descriptive and do not identify causality.
- Completed-episode logs omit unfinished episode fragments at rollout boundaries.


In [1]:
from pathlib import Path
import csv
import json

ROOT = Path(globals().get("_REPO_ROOT", Path.cwd()))
OUT = ROOT / "analysis_results" / "gae_diagnostic"
summary = json.loads((OUT / "diagnosis_summary.json").read_text())

def load_csv(name):
    with (OUT / name).open() as handle:
        return list(csv.DictReader(handle))

windows = load_csv("window_diagnostics.csv")
lambda_sensitivity = load_csv("lambda_sensitivity.csv")
collision_times = load_csv("collision_time_distributions.csv")
correlations = load_csv("proxy_correlations.csv")
availability = load_csv("data_availability.csv")
print(summary["verdict"])


No GAE correctness defect found. Existing evidence does not justify changing lambda from 0.995 yet; collect transition-level counterfactual telemetry before a 0.99 ablation.


## Data

### 1. Late-training advantage-energy proxy and critic fit

The collision/ordinary split is at the transition role, not the completed episode outcome.


In [2]:
for row in windows:
    print(row)


{'window': 'U2-U30', 'updates': '29', 'overall_advantage_second_moment_proxy': '0.09634231719061366', 'overall_advantage_rms_proxy': '0.3103905881153835', 'collision_advantage_second_moment_proxy': '0.1562696433457097', 'collision_advantage_rms_proxy': '0.39530955382549215', 'ordinary_advantage_second_moment_proxy': '0.036414991035517624', 'ordinary_advantage_rms_proxy': '0.19082712342724664', 'collision_to_ordinary_second_moment_ratio': '4.291354711395946', 'overall_ev_pre_mean': '0.7435231283001278', 'overall_ev_post_mean': '0.8811870306112541', 'collision_ev_pre_mean': '0.6748172361557434', 'collision_ev_post_mean': '0.8610018270217364', 'ordinary_ev_pre_mean': '0.7706238278318768', 'ordinary_ev_post_mean': '0.8507595887246666', 'mean_approx_kl': '0.06804283189072081', 'max_approx_kl': '0.4431079248370952', 'mean_actor_grad_norm': '31.373074993491173'}
{'window': 'U10-U30', 'updates': '21', 'overall_advantage_second_moment_proxy': '0.08653251457948352', 'overall_advantage_rms_proxy'

## Results

### 2. Collision-role raw advantage energy is larger, while critic post-fit remains strong

At U20-U30, collision/ordinary second-moment proxies are 0.1435 and 0.0246. Overall post-update
explained variance is 0.925 and collision-role post-update explained variance is 0.913. This is
consistent with harder collision targets that the critic can fit, not with critic failure.


In [3]:
late = next(row for row in windows if row["window"] == "U20-U30")
print("U20-U30 role ratio:", late["collision_to_ordinary_second_moment_ratio"])
print("U20-U30 overall/collision post EV:", late["overall_ev_post_mean"], late["collision_ev_post_mean"])


U20-U30 role ratio: 5.832152114901845
U20-U30 overall/collision post EV: 0.9254303617150286 0.9131333003491572


### 3. Lambda 0.99 materially shortens the credit path

With gamma fixed at 0.999, changing lambda from 0.995 to 0.99 changes the two-second weight from
0.300 to 0.110 and the four-second weight from 0.090 to 0.012. This can reduce long-tail variance,
but also removes direct collision credit that must then come from critic bootstrap.


In [4]:
for row in lambda_sensitivity:
    print(row)


{'gae_lambda': '0.99', 'gamma_times_lambda': '0.98901', 'td_residual_half_life_s': '0.6272350515610913', 'geometric_horizon_s': '0.9099181073703321', 'weight_1s': '0.33118318797366136', 'weight_2s': '0.10968230399639753', 'weight_4s': '0.01203020780995816', 'weight_6s': '0.0013195009101516666', 'weight_8s': '0.00014472589995077832'}
{'gae_lambda': '0.995', 'gamma_times_lambda': '0.994005', 'td_residual_half_life_s': '1.1527395991034204', 'geometric_horizon_s': '1.6680567139282811', 'weight_1s': '0.5480963338904562', 'weight_2s': '0.30040959122415845', 'weight_4s': '0.09024592249946599', 'weight_6s': '0.027110740687711663', 'weight_8s': '0.00814432652777962'}
{'gae_lambda': '0.9975', 'gamma_times_lambda': '0.9965025000000001', 'td_residual_half_life_s': '1.978368353430495', 'geometric_horizon_s': '2.8591851322373687', 'weight_1s': '0.7044322955008794', 'weight_2s': '0.49622485894463825', 'weight_4s': '0.2462391106346261', 'weight_6s': '0.1221899679413205', 'weight_8s': '0.06063369960613

### 4. Current proxies do not explain actor instability spikes

The overall second-moment proxy has Pearson correlation -0.204 with mean KL and 0.071 with mean
actor gradient norm; collision-role values are -0.148 and 0.020. Advantage normalization and the
small update count limit interpretation, but there is no positive association supporting an immediate
lambda reduction.


In [5]:
for row in correlations:
    if row["source"] in {"overall advantage second-moment proxy", "collision advantage second-moment proxy"}:
        print(row)


{'updates': 'U2-U30', 'source': 'overall advantage second-moment proxy', 'target': 'actor grad norm mean', 'pearson_r': '0.07053376699776398', 'spearman_rho': '0.07044334975369458'}
{'updates': 'U2-U30', 'source': 'overall advantage second-moment proxy', 'target': 'actor grad norm max', 'pearson_r': '0.108447126066891', 'spearman_rho': '0.007881773399014778'}
{'updates': 'U2-U30', 'source': 'overall advantage second-moment proxy', 'target': 'approx KL mean', 'pearson_r': '-0.20352945595766525', 'spearman_rho': '-0.11724137931034483'}
{'updates': 'U2-U30', 'source': 'overall advantage second-moment proxy', 'target': 'clip fraction mean', 'pearson_r': '0.11061443613021066', 'spearman_rho': '0.10492610837438424'}
{'updates': 'U2-U30', 'source': 'collision advantage second-moment proxy', 'target': 'actor grad norm mean', 'pearson_r': '0.019704666866852412', 'spearman_rho': '0.08571428571428572'}
{'updates': 'U2-U30', 'source': 'collision advantage second-moment proxy', 'target': 'actor gra

### 5. Episode timing confirms broad failures but cannot reconstruct collision lag

The collision-time distribution spans early and late failures. Existing episode rows do not contain
the rollout timestep and environment rank needed to align advantages at 0.5/1/2/4 seconds before impact.


In [6]:
for row in collision_times:
    print(row)
print("Evidence availability:")
for row in availability:
    print(row)


{'window': 'U1-U30', 'collision_episodes': '1327', 'mean_collision_time_s': '3.810723436322491', 'p05_collision_time_s': '0.9700000000000006', 'p25_collision_time_s': '1.9800000000000015', 'median_collision_time_s': '3.4899999999999696', 'p75_collision_time_s': '5.699999999999923', 'p95_collision_time_s': '7.469999999999885'}
{'window': 'U10-U30', 'collision_episodes': '845', 'mean_collision_time_s': '3.7700710059171194', 'p05_collision_time_s': '0.9500000000000006', 'p25_collision_time_s': '1.9000000000000015', 'median_collision_time_s': '3.3899999999999717', 'p75_collision_time_s': '5.709999999999923', 'p95_collision_time_s': '7.479999999999885'}
{'window': 'U20-U30', 'collision_episodes': '423', 'mean_collision_time_s': '3.6465011820330586', 'p05_collision_time_s': '0.9400000000000006', 'p25_collision_time_s': '1.8300000000000014', 'median_collision_time_s': '3.1999999999999758', 'p75_collision_time_s': '5.319999999999931', 'p95_collision_time_s': '7.429999999999886'}
Evidence avail

## Takeaways

1. Keep lambda at 0.995 for the 45-update reproduction run.
2. Before any buffer `get()` call, read raw `[time, env]` advantages without model forward or RNG use.
3. Reconstruct current TD residuals and recompute 0.99/0.995/0.9975 advantages on the same rollout.
4. Compare role tails, sign flips, rank/cosine agreement, and collision-lag slices before running 0.99.
5. Do not test 0.9975 or 1.0 now; no current evidence indicates credit is too short.
